### Install new libraries

In [1]:
!pip install ddgs trafilatura -q

In [1]:
!pip install --upgrade ddgs trafilatura requests urllib3 -q

### Setup

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from IPython.display import Markdown, display
from tqdm import tqdm
import contextlib
import io

from ddgs import DDGS
import trafilatura

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing !!")

client = OpenAI()

MODEL = 'gpt-4.1-mini'

/Users/rajan/Desktop/Projects/ai-engineering/ai_env/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


### Step 1: Define the Tools

In [3]:
def search_web(query):
    """Search the Web using DuckDuckGo browser. Returns 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 Got results")
    return json.dumps(results, indent=2)

In [4]:
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura"""
    downloaded = trafilatura.fetch_url(url=url)
    if downloaded:
        text = trafilatura.extract(downloaded)
        if text:
            print(f"  \u2705 Got text: {len(text)} chars")
            return text
    print(f"  \u274c Failed to fetch or extract text")
    return f"Could not extract text from {url}. Try a different source."

In [5]:
search_web("Ai in healthcare in 2030")

  ✅ Got results


'[\n  {\n    "title": "Pulse Report: Healthcare in 2030: How AI Agents Promote ...",\n    "href": "https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/",\n    "body": "With the rapid democratization of AI , the future of healthcare was already under redevelopment. Now, in an era of AI agents, that future is getting reshaped. What might this new healthcare future, powered by AI agents, look like?"\n  },\n  {\n    "title": "Artificial Intelligence in Healthcare: A Narrative Review of ...",\n    "href": "https://pmc.ncbi.nlm.nih.gov/articles/PMC12764347/",\n    "body": "To bridge the implementation gap, this review examines areas where AI is making a difference in healthcare today and provides evidence-based strategies for healthcare leaders to deploy these technologies safely and equitably."\n  },\n  {\n    "title": "AI and Healthcare in 2030: Predictions and Pathways",\n    "href": "https://japmi.org/index.php

In [6]:
fetch_url("https://www.aiprm.com/en-gb/ai-in-healthcare-statistics/")

  ✅ Got text: 15956 chars


"A Deloitte survey found that about half (53%) of consumers believe generative AI will improve access to healthcare, while a further 46% think AI will make healthcare more affordable for ordinary Americans.\nWith this in mind, AIPRM felt it was the perfect time to produce the definitive page on AI in healthcare statistics, including the perceptions of healthcare providers and patients, alongside global and national revenue from AI in the healthcare sector.\nAI in healthcare statistics indicate that the market has grown considerably and shows no sign of slowing down. As of 2024, the AI in healthcare market stood at $32.3 billion.\nAI in the healthcare market has exploded in value since 2016. Between 2016 and 2023, its market size grew from $1.1 billion to $22.4 billion, representing a rise of 1,779%.\nBetween 2022 and 2023 alone, the market grew by close to half (45%) from $15.4 billion to $22.4 billion.\nThe single biggest increase in a single year came between 2019 and 2020. The marke

### Describe as LLM tools

In [7]:
tools = []

search_web_function = {
    "name": "search_web",
    "description": "Search the Web using DuckDuckGo browser. Returns 3 results.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find relevant url or websites."
            }
        },
        "required": ['query']
    }
}



fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch and extract the main text content from a web page.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL of the web page to fetch and extract text from."
            }
        },
        "required": ['url']
    }
}


tools.extend([
    {"type": "function", "function": search_web_function},
    {"type": "function", "function": fetch_url_function}
    ])

### Tool call handler

In [8]:
def handle_tool_call(tool_calls):
    tool_call_results = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        print(f"  \U0001f527 Calling function {function_name} with argument: {args}") # For future debugging

        # Route to the appropriate function based on function_name
        if function_name == "search_web":
            result = search_web(args['query'])
            content = f"Search results: {result}"
        

        elif function_name == "fetch_url":
            result = fetch_url(args['url'])
            content = f"Fetched URL content: {result}"

        else:
            content = f"Unknown function: {function_name}"

        tool_call_result = {
            "role" : "tool",
            "tool_call_id" : tool_call.id,
            "content" : content
        }
        tool_call_results.append(tool_call_result)
    return tool_call_results

### Step 4: The system prompt

This tells the LLM who it is and how to behave. The key things:

* What its job is
* What tools it has
* What process to follow
* What output format to produce

In [9]:
RESEARCH_AGENT_PROMPT = """You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps?
5. If there are gaps, search again with a different query
6. When you have enough information from at least 6 different sources, synthesize into a research brief

You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 6 sources, keep searching.

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that "DONE:" should be at the start of the final response, so that it can be easily parsed and extracted.
You CANNOT and SHOULD NOT include "DONE:" in nay other part of your response except at the very beginning of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Step 5: The Agentic Loop

In [10]:
def run_research_agent(topic: str, max_iterations: int = 10) -> str:
    """ 
    Run the research agnent on a topic and return the research brief

    Args:
        topic: The topic to research
        max_iterations: Safety limit to prevent infinite loops

    Returns:
    The research brief as a string
    
    """

    print(f"\n\U0001F50D Starting research on: {topic}")
    print("="*60)

    # Initialize the conversation messages list with system prompt + research task

    messages = [{"role" : "system", "content" : RESEARCH_AGENT_PROMPT},
                {"role" : "user", "content" : f"Research the following topic and produce a comprehensive research brief:\n {topic}"}]

    # Loop

    interation = 0
    while interation < max_iterations:
        interation+=1
        print(f"\n--- iteration {interation} ---")

        #1. Call the LLM and get response
        response = client.chat.completions.create(
            model = MODEL,
            messages=messages,
            tools=tools
        )

        message = response.choices[0].message
        messages.append(message)

        #2. Check if LLM called tolls

        if message.tool_calls:
            tool_results = handle_tool_call(message.tool_calls)
            messages.extend(tool_results)

        #3. Otherwise: no tolls were called, read message content
        else:
            content = message.content #No tools calls, just a normal message

            # check if DONE, then return
            if content.startswith("DONE:"):
                research_brief = content[len("DONE:"):].strip()
                print(f"\n\u2705 Research complete!")
                return research_brief

            # Otherwise: not yet done
            else:
                print("  \U0001F4AD Agent is thinking:")
                pprint(content)
                # Loop continues to next iteration

        #4. If we're entering the final iteration , force a final answer
        if (interation == max_iterations -1):
            print(f"  \u26a0 Safety limit reached. Stopping research in next iteration.")
            messages.append({"role" : "user", "content" : "You have reached the maximum number of iterations. Please deliver your research brief now. You MUST respond with DONE: followed by your brief"})


    # Fallback return
    return "Research incomplete. Maximum iterations reached without finalizing brief."


### Lets run it !!!

In [11]:
#brief = run_research_agent("AI in healthcare in 2030")
#display(Markdown(brief))

### Evals

In [12]:
### Evals

JUDGE_PROMPT = """

You are scoring a research agent's **deliverable research brief** to determine if there was a **source breadth failure**.
Return **only** `TRUE` or `FALSE`.

## Definitions

**Source:** A distinct origin of information referenced in the brief — typically a URL, article, or named publication/site that the agent pulled information from. Count each unique source once, regardless of how many times it's referenced or how many facts are drawn from it.

**Note:** We are NOT testing correct formal citation style, footnote formatting, or whether sources are properly attributed per claim. We are only testing the breadth of research — whether the agent actually consulted and drew from a sufficiently wide set of sources before producing its brief.

**Source breadth failure (label TRUE):** Any of the below occurred:

1. **Fewer than 6 distinct sources:** The brief references, cites, or draws from fewer than 6 unique sources in total.
2. **Duplicate/near-duplicate sources counted as distinct:** The same source is referenced multiple times under different names, URLs, or phrasings, and the *actual* distinct source count (after deduplication) is below 6 — even if the brief superficially lists 6+ references.
3. **Sources mentioned but not substantively used:** A source is listed (e.g., in a references/sources section) but no actual information, fact, or claim in the brief body is attributable to it. Such sources do not count toward the 6.
4. **Vague or unverifiable sourcing:** The brief references information generically (e.g., "various reports suggest," "industry sources indicate") without identifying an actual distinct source. Such references do not count toward the 6.

**No source breadth failure (label FALSE):** The brief draws substantive content from at least 6 distinct, identifiable sources, each contributing to the brief's actual claims or content (not merely listed).

## Output Format

Return exactly one token: `TRUE` or `FALSE`. No explanations.

"""

In [13]:
EVAL_TOPICS = [
    "The economic impact of quantum computing by 2035",
    "Recent advances in CRISPR gene editing therapies",
    "The rise of electric vehicle adoption in Europe",
    "History and cultural impact of the Silk Road",
    "Current state of fusion energy research",
    "Trends in global coffee supply chains",
    "The effects of social media on teenage mental health",
    "Space tourism industry developments in 2025",
    "Renewable energy policy changes in the US",
    "The future of remote work post-pandemic",
]

In [14]:
JUDGE_MODEL = "gpt-4.1"
MODEL = "gpt-4.1-nano"


def judge_brief(brief: str) -> str:
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": brief},
        ],
        max_tokens=1,
        temperature=0
    )
    return response.choices[0].message.content.strip()


def run_eval(topic: str) -> dict:
    """Run one topic through the agent + judge. Suppresses the agent's internal
    tool-call logs, but catches and surfaces any failure (e.g. the intermittent
    SSL error from search_web) instead of letting it kill the whole batch."""
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            brief = run_research_agent(topic)
        verdict = judge_brief(brief)
        return {"topic": topic, "brief": brief, "verdict": verdict, "error": None}
    except Exception as e:
        return {"topic": topic, "brief": None, "verdict": "ERROR", "error": f"{type(e).__name__}: {e}"}


results = []
progress = tqdm(EVAL_TOPICS, desc="Evaluating")
for topic in progress:
    progress.set_postfix_str(topic[:40])
    result = run_eval(topic)
    results.append(result)
    status = result["verdict"] if result["error"] is None else f"ERROR: {result['error']}"
    tqdm.write(f"{status:<40} | {topic}")

errors = sum(r["verdict"] == "ERROR" for r in results)
failures = sum(r["verdict"] == "TRUE" for r in results)
print(f"\n{failures}/{len(results)} source-breadth failures, {errors} runs errored out")

Evaluating:  10%|█         | 1/10 [00:06<00:56,  6.24s/it, Recent advances in CRISPR gene editing t]

TRUE                                     | The economic impact of quantum computing by 2035


Evaluating:  20%|██        | 2/10 [00:14<01:00,  7.57s/it, The rise of electric vehicle adoption in]

TRUE                                     | Recent advances in CRISPR gene editing therapies


Evaluating:  30%|███       | 3/10 [00:15<00:31,  4.54s/it, History and cultural impact of the Silk ]

...

FALSE                                    | Space tourism industry developments in 2025


Evaluating:  90%|█████████ | 9/10 [00:59<00:06,  6.38s/it, The future of remote work post-pandemic] 

ERROR: ValueError: Unsupported protocol version 0x304 | Renewable energy policy changes in the US


Evaluating: 100%|██████████| 10/10 [01:08<00:00,  6.88s/it, The future of remote work post-pandemic]

TRUE                                     | The future of remote work post-pandemic

5/10 source-breadth failures, 3 runs errored out
